# Data Integration Phase
Enriches data from silver lake and writes aggregated and enriched taxi_trips data as gold Delta table `integrated_taxi_trips`.


## 1. Configure Spark


In [26]:
import os
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
if not (ROOT / "src").is_dir():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.spark import create_spark, project_root

spark = create_spark("integration-gold")
ROOT = project_root()

from src.lake import GOLD, SILVER, read_delta, show_delta, write_gold

print("JAVA_HOME:", os.environ["JAVA_HOME"])
print("PySpark:", spark.version)
print("ROOT:", ROOT)


JAVA_HOME: /opt/homebrew/opt/openjdk@17/libexec/openjdk.jdk/Contents/Home
PySpark: 3.5.4
ROOT: /Users/samuelflodin/programmering/uppgifter/5an/bigdata/id2221-labs


## 2. Load silver tables

Read silver delta tables.

In [27]:
from pyspark.sql import functions as F

trips = read_delta(spark, SILVER / "taxi_trips")
weather = read_delta(spark, SILVER / "weather")
air_quality = read_delta(spark, SILVER / "air_quality")
zones = read_delta(spark, SILVER / "taxi_zones")

## 3. Hourly weather (NYC local)



In [28]:
hourly_weather = (
    weather
    .groupBy(
        F.col("observation_date").alias("pickup_date"),
        F.col("observation_hour").alias("pickup_hour"),
    )
    .agg(
        F.avg("temperature_c").alias("temperature_c"),
        F.avg("wind_speed_ms").alias("wind_speed_ms"),
    )
)

print(f"hourly_weather: {hourly_weather.count():,} hours")
hourly_weather.show(3, truncate=False)


hourly_weather: 8,784 hours
+-----------+-----------+-------------+------------------+
|pickup_date|pickup_hour|temperature_c|wind_speed_ms     |
+-----------+-----------+-------------+------------------+
|2024-08-23 |2          |22.2         |3.1111111111111107|
|2024-08-23 |7          |20.6         |3.611111111111111 |
|2024-08-23 |11         |18.9         |0.0               |
+-----------+-----------+-------------+------------------+
only showing top 3 rows



## 4. Hourly air quality (NYC local)



In [29]:
NYC_COUNTIES = [5, 47, 61, 81, 85]

hourly_aq = (
    air_quality
    .filter(F.col("county_code").isin(NYC_COUNTIES))
    .groupBy(
        F.col("measurement_date").alias("pickup_date"),
        F.col("measurement_hour").alias("pickup_hour"),
    )
    .agg(
        F.avg("value").alias("pm25"),
        F.first("unit").alias("pm25_unit"),
    )
)

print(f"hourly_aq: {hourly_aq.count():,} hours")
hourly_aq.show(3, truncate=False)


hourly_aq: 8,783 hours
+-----------+-----------+------------------+---------------------------+
|pickup_date|pickup_hour|pm25              |pm25_unit                  |
+-----------+-----------+------------------+---------------------------+
|2024-01-01 |0          |14.520000000000001|Micrograms/cubic meter (LC)|
|2024-01-01 |1          |14.459999999999999|Micrograms/cubic meter (LC)|
|2024-01-01 |2          |14.440000000000001|Micrograms/cubic meter (LC)|
+-----------+-----------+------------------+---------------------------+
only showing top 3 rows



## 5. Pickup / dropoff zone lookups

`taxi_zones` is a small table ==> we can easily and with minimal overhead split them into `pickup_zones` and `dropoff_zones`


In [30]:
pickup_zones = zones.select(
    F.col("location_id").alias("pickup_location_id"),
    F.col("zone").alias("pickup_zone"),
    F.col("borough").alias("pickup_borough"),
)

dropoff_zones = zones.select(
    F.col("location_id").alias("dropoff_location_id"),
    F.col("zone").alias("dropoff_zone"),
    F.col("borough").alias("dropoff_borough"),
)


## 6. Enrich trips and write `integrated_taxi_trips`

Left-join weather and air_quality data onto taxi_trip data.

In [31]:
integrated = (
    trips
    .join(F.broadcast(pickup_zones), "pickup_location_id", "left")
    .join(F.broadcast(dropoff_zones), "dropoff_location_id", "left")
    .join(F.broadcast(hourly_weather), ["pickup_date", "pickup_hour"], "left")
    .join(F.broadcast(hourly_aq), ["pickup_date", "pickup_hour"], "left")
    .fillna(
        {
            "pickup_zone": "UNKNOWN",
            "pickup_borough": "UNKNOWN",
            "dropoff_zone": "UNKNOWN",
            "dropoff_borough": "UNKNOWN",
        }
    )
    .select(
        "taxi_type",
        "vendor_id",
        "pickup_datetime",
        "dropoff_datetime",
        "passenger_count",
        "trip_distance",
        "pickup_location_id",
        "pickup_zone",
        "pickup_borough",
        "dropoff_location_id",
        "dropoff_zone",
        "dropoff_borough",
        "fare_amount",
        "tip_amount",
        "tolls_amount",
        "total_amount",
        "temperature_c",
        "wind_speed_ms",
        "pm25",
        "pm25_unit",
        "pickup_date",
        "pickup_hour",
    )
    .cache()
)

# Materialize the shared input once so both writes measure partitioning and I/O.
integrated.count()

import time

t0 = time.perf_counter()
write_gold(integrated, "integrated_taxi_trips", partition_by=["pickup_date"])
print(f"write by_date: {time.perf_counter() - t0:.1f}s")
show_delta(spark, GOLD / "integrated_taxi_trips")

write by_date: 25.7s


integrated_taxi_trips: 9417383 rows @ data/lake/gold/integrated_taxi_trips
+---------+---------+-------------------+-------------------+---------------+-------------+------------------+---------------------+--------------+-------------------+-----------------------+---------------+-----------+----------+------------+------------+-------------+-----------------+------------------+---------------------------+-----------+-----------+
|taxi_type|vendor_id|pickup_datetime    |dropoff_datetime   |passenger_count|trip_distance|pickup_location_id|pickup_zone          |pickup_borough|dropoff_location_id|dropoff_zone           |dropoff_borough|fare_amount|tip_amount|tolls_amount|total_amount|temperature_c|wind_speed_ms    |pm25              |pm25_unit                  |pickup_date|pickup_hour|
+---------+---------+-------------------+-------------------+---------------+-------------+------------------+---------------------+--------------+-------------------+-----------------------+--------------

## 7. Two storage designs

To compare two storage designs, we can partition the same rows aggregated from silver to gold layer in two different ways:

| Partitioning Logic | Table | Partition | Suited for |
| --- | --- | --- | --- |
| By timestamp | `integrated_taxi_trips` | `pickup_date` | average duration per day |
| By location | `integrated_taxi_trips_by_borough` | `pickup_borough` | trip data per borough location |


In [32]:
import time

t0 = time.perf_counter()
write_gold(integrated, "integrated_taxi_trips_by_borough", partition_by=["pickup_borough"])
print(f"write by_borough: {time.perf_counter() - t0:.1f}s")


def storage_report(table_name: str) -> None:
    path = GOLD / table_name
    files = [f for f in path.rglob("*.parquet") if f.is_file()]
    size_mb = sum(f.stat().st_size for f in files) / (1024 * 1024)
    n_parts = len({f.parent for f in files})
    print(
        f"{table_name:40} files={len(files):>5}  partitions={n_parts:>4}  size={size_mb:>8.1f} MB"
    )


print()
print("Storage")
storage_report("integrated_taxi_trips")
storage_report("integrated_taxi_trips_by_borough")

write by_borough: 35.3s

Storage
integrated_taxi_trips                    files=  222  partitions=  92  size=   227.7 MB
integrated_taxi_trips_by_borough         files=  200  partitions=   8  size=   226.9 MB


### Queries on both designs


In [33]:
import time


def queries(df):
    duration_min = (
        F.unix_timestamp("dropoff_datetime") - F.unix_timestamp("pickup_datetime")
    ) / 60.0
    return {
        "trips per borough": (
            df.groupBy("pickup_borough")
            .agg(F.count(F.lit(1)).alias("trips"))
            .orderBy(F.desc("trips"))
        ),
        "avg duration per day": (
            df.withColumn("duration_min", duration_min)
            .groupBy("pickup_date")
            .agg(F.avg("duration_min").alias("avg_duration_min"))
            .orderBy("pickup_date")
        ),
        "avg fare per borough": (
            df.groupBy("pickup_borough")
            .agg(F.avg("fare_amount").alias("avg_fare"))
            .orderBy("pickup_borough")
        ),
    }


def run_queries(table_name: str) -> None:
    spark.catalog.clearCache()
    df = read_delta(spark, GOLD / table_name)
    print(f"\n{table_name}")
    print("-" * 40)
    for name, q in queries(df).items():
        t0 = time.perf_counter()
        rows = q.collect()
        elapsed = time.perf_counter() - t0
        print(f"\n{name}  ({elapsed:.2f}s, {len(rows):,} rows)")
        spark.createDataFrame(rows).show(20, truncate=False)


run_queries("integrated_taxi_trips")
run_queries("integrated_taxi_trips_by_borough")



integrated_taxi_trips
----------------------------------------



trips per borough  (2.37s, 8 rows)


+--------------+-------+
|pickup_borough|trips  |
+--------------+-------+
|Manhattan     |8442728|
|Queens        |817017 |
|Brooklyn      |95738  |
|Unknown       |31298  |
|Bronx         |24715  |
|N/A           |4752   |
|EWR           |915    |
|Staten Island |220    |
+--------------+-------+




avg duration per day  (9.31s, 92 rows)
+-----------+------------------+
|pickup_date|avg_duration_min  |
+-----------+------------------+
|2024-01-01 |16.444424507823307|
|2024-01-02 |16.906788016430834|
|2024-01-03 |16.545877362397537|
|2024-01-04 |16.08400669921135 |
|2024-01-05 |15.385487487153334|
|2024-01-06 |14.58333002828419 |
|2024-01-07 |13.901090108635216|
|2024-01-08 |15.62057394556443 |
|2024-01-09 |15.068740392765223|
|2024-01-10 |15.230732569936643|
|2024-01-11 |16.433418924474225|
|2024-01-12 |16.471563084834628|
|2024-01-13 |15.074412049124728|
|2024-01-14 |14.352769278119478|
|2024-01-15 |14.90966166434527 |
|2024-01-16 |16.602465760869592|
|2024-01-17 |16.343770248792744|
|2024-01-18 |16.05834059067091 |
|2024-01-19 |15.139623995861996|
|2024-01-20 |14.266279737782696|
+-----------+------------------+
only showing top 20 rows




avg fare per borough  (1.55s, 8 rows)
+--------------+------------------+
|pickup_borough|avg_fare          |
+--------------+------------------+
|Bronx         |31.733360712118152|
|Brooklyn      |28.560865487058393|
|EWR           |90.21594535519125 |
|Manhattan     |15.430819608307019|
|N/A           |87.23950968013462 |
|Queens        |51.962818815275156|
|Staten Island |42.360727272727274|
|Unknown       |20.224147868873448|
+--------------+------------------+


integrated_taxi_trips_by_borough
----------------------------------------



trips per borough  (2.89s, 8 rows)
+--------------+-------+
|pickup_borough|trips  |
+--------------+-------+
|Manhattan     |8442728|
|Queens        |817017 |
|Brooklyn      |95738  |
|Unknown       |31298  |
|Bronx         |24715  |
|N/A           |4752   |
|EWR           |915    |
|Staten Island |220    |
+--------------+-------+




avg duration per day  (2.08s, 92 rows)
+-----------+------------------+
|pickup_date|avg_duration_min  |
+-----------+------------------+
|2024-01-01 |16.44442450782354 |
|2024-01-02 |16.906788016430923|
|2024-01-03 |16.54587736239775 |
|2024-01-04 |16.08400669921109 |
|2024-01-05 |15.38548748715337 |
|2024-01-06 |14.583330028284262|
|2024-01-07 |13.901090108635143|
|2024-01-08 |15.620573945564523|
|2024-01-09 |15.068740392765537|
|2024-01-10 |15.23073256993671 |
|2024-01-11 |16.433418924474434|
|2024-01-12 |16.47156308483458 |
|2024-01-13 |15.074412049124797|
|2024-01-14 |14.352769278119418|
|2024-01-15 |14.90966166434513 |
|2024-01-16 |16.6024657608695  |
|2024-01-17 |16.34377024879265 |
|2024-01-18 |16.058340590670788|
|2024-01-19 |15.139623995861927|
|2024-01-20 |14.266279737782732|
+-----------+------------------+
only showing top 20 rows


avg fare per borough  (0.81s, 8 rows)
+--------------+------------------+
|pickup_borough|avg_fare          |
+--------------+---------------

26/09/11 16:29:08 WARN HeartbeatReceiver: Removing executor driver with no recent heartbeats: 566556 ms exceeds timeout 120000 ms
26/09/11 16:29:08 WARN SparkContext: Killing executors is not supported by current scheduler.
26/09/11 16:29:08 ERROR Inbox: Ignoring error
org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:56)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:310)
	at org.apache.spark.rpc.RpcTimeout.awaitResult(RpcTimeout.scala:75)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRefByURI(RpcEnv.scala:102)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRef(RpcEnv.scala:110)
	at org.apache.spark.util.RpcUtils$.makeDriverRef(RpcUtils.scala:36)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.driverEndpoint$lzycompute(BlockManagerMasterEndpoint.scala:124)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.org$apache$spark$storage$BlockManagerMasterEndpoint$$